# Hotel search ranking with learning to rank (Expedia data)

Run the cells top to bottom (**Runtime → Run all**). No GPU needed; the standard CPU runtime is fine.

Before you start, one time only:
1. Sign in at kaggle.com and open https://www.kaggle.com/competitions/expedia-personalized-sort/rules, then click **I Understand and Accept**. Kaggle only lets you download competition data after accepting the rules.
2. Have your Kaggle username and an API key ready (kaggle.com → Settings → API → Create New Token). Paste them only into the prompt in step 2, never into a shared cell or screenshot.

## 1. Get the code

In [ ]:
!git clone -q https://github.com/gyanasaiadusumilli/hotel-search-ranking.git
%cd hotel-search-ranking
!pip install -q -r requirements.txt kagglehub

## 2. Sign in to Kaggle and download the data (about 1 GB compressed)

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
import glob, os, zipfile
path = kagglehub.competition_download("expedia-personalized-sort")
for z in glob.glob(os.path.join(path, "**", "*.zip"), recursive=True):
    zipfile.ZipFile(z).extractall(os.path.dirname(z))
TRAIN = [f for f in glob.glob(os.path.join(path, "**", "train.csv"), recursive=True)][0]
print(TRAIN, round(os.path.getsize(TRAIN) / 1e9, 2), "GB")

## 3. Train and evaluate

Uses half of the ~400k searches (`--frac 0.5`) to fit in Colab's memory. Takes roughly 15 to 30 minutes.

In [ ]:
!python -m src.run --data "$TRAIN" --frac 0.5

## 4. Results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display, Markdown
t = pd.read_csv("results/comparison.csv")
display(Markdown("### All test searches")); display(t[t.subset == "all_test"])
display(Markdown("### Searches shown in random order (unbiased by position)")); display(t[t.subset == "random_order_searches"])
display(Markdown("### Searches shown in Expedia's order")); display(t[t.subset == "expedia_ordered_searches"])
print(json.dumps(json.load(open("results/ab_tests.json")), indent=2))
display(Image("results/comparison_ndcg.png")); display(Image("results/feature_importance.png"))

## 5. Download the results
Send `results.zip` back so the README can be updated with the real numbers.

In [ ]:
!zip -qr results.zip results models
from google.colab import files
files.download("results.zip")